In [ ]:
import pandas as pd
from p_2_transform import get_toyota_data

from sklearn.preprocessing import MinMaxScaler
from IPython.display import HTML

In [ ]:
data = get_toyota_data()

In [ ]:
def calculate_impuesto_matriculacion(value: float):
    if value <= 120:
        return 0
    elif value <= 160:
        return 0.0475
    elif value <= 200:
        return 0.0975
    else:
        return 0.1475

In [ ]:
import datetime

df = pd.DataFrame(data)

df = df.loc[
    (df["previous_owner"] == "Particular") & (df["vehicle_status"] == "Disponible")
]

del df["vehicle_status"]
del df["previous_owner"]
del df["vin"]
del df["version"]


# df['impuesto_matriculacion'] = 

df["production_date"] = pd.to_datetime(df["production_date"])
df["age"] = datetime.datetime.today() - df["production_date"]
df["age"] = df["age"].map(lambda x: x.days / 365)
df.dropna(subset="age", inplace=True)
df.dropna(subset="km", inplace=True)
df["km_per_year"] = df["km"] / df["age"]
df["km_per_year"] = df["km_per_year"].astype("int")
df = df.loc[df.price <= 30_000]
df = df.loc[df.km_per_year <= 20_000]

scaler = MinMaxScaler()
df[["price_score"]] = scaler.fit_transform(df[["price"]])
df[["age_score"]] = scaler.fit_transform(df[["age"]])
df[["km_per_year_score"]] = scaler.fit_transform(df[["km_per_year"]])

df["score"] = df.price_score + df.km_per_year_score + df.age_score * 0.3
df = df.sort_values(by="score")

df = df.loc[
    ~df.province.isin(
        ("BARCELONA",),
    )
]
df = df.loc[
    ~df.city.isin(
        ("SABADELL",),
    )
]

HTML(df.head(50).to_html(escape=False))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


def filter_df(df_raw: pd.DataFrame, model: str) -> pd.DataFrame:
    df = df_raw.loc[df_raw.model == model].copy()
    df = df[
        ["url", "price", "km", "year", "price_score", "age_score", "km_per_year_score"]
    ]
    return df


def display_cars(df: pd.DataFrame):
    sns.pairplot(data=df)
    plt.show()
    display(sns.scatterplot(data=df, x="km", y="price", hue="year"))
    plt.grid()
    plt.show()

In [ ]:
display_cars(df)

In [ ]:
import plotly.express as px
from IPython.display import HTML

fig = px.scatter(
    df,
    x="km",
    y="price",
    custom_data=["url"],
    title="Car Prices vs Kilometers",
)

fig.update_traces(
    mode="markers",
    marker=dict(size=10),
    hovertemplate="KM: %{x}<br>Price: %{y}<br>URL: %{customdata[0]}<extra></extra>",
)
fig.update_layout(clickmode="event+select", width=1280, height=1024)

div_id = "carplot"

html = fig.to_html(include_plotlyjs="cdn", full_html=False, div_id=div_id)

js = f"""
<div id="{div_id}-hover-panel" style="
  font-family: sans-serif; margin-top:6px; padding:8px 10px; 
  border:1px solid #ccc; border-radius:6px; display:none; max-width:100%;
  background:#fff; box-shadow:0 2px 8px rgba(0,0,0,0.08);">
  <strong>Point info</strong><br>
  <span id="{div_id}-km"></span> | <span id="{div_id}-price"></span><br>
  <a id="{div_id}-link" href="#" target="_blank" rel="noopener noreferrer"></a>
  <div style="margin-top:4px; font-size:12px; color:#666;">
    Click on a point to pin/unpin. Press Esc to clear.
  </div>
</div>

<script>
(function() {{
  const gd = document.getElementById("{div_id}");
  const panel = document.getElementById("{div_id}-hover-panel");
  const kmEl = document.getElementById("{div_id}-km");
  const priceEl = document.getElementById("{div_id}-price");
  const linkEl = document.getElementById("{div_id}-link");
  let pinned = false;

  function showInfo(pt) {{
    const x = pt.x;
    const y = pt.y;
    const url = (pt.customdata && pt.customdata[0]) ? pt.customdata[0] : "";
    kmEl.textContent = "KM: " + x;
    priceEl.textContent = "Price: " + y;
    linkEl.textContent = url;
    linkEl.href = url || "#";
    panel.style.display = "block";
  }}

  gd.on('plotly_hover', function(ev) {{
    if (!ev || !ev.points || !ev.points.length) return;
    if (!pinned) showInfo(ev.points[0]);
  }});

  gd.on('plotly_unhover', function() {{
    if (!pinned) panel.style.display = "none";
  }});

  gd.on('plotly_click', function(ev) {{
    if (!ev || !ev.points || !ev.points.length) return;
    // toggle pin; always refresh content to the clicked point
    pinned = !pinned;
    showInfo(ev.points[0]);
  }});

  window.addEventListener('keydown', function(e) {{
    if (e.key === 'Escape') {{
      pinned = false;
      panel.style.display = "none";
    }}
  }});
}})();
</script>
"""

HTML(html + js)
